# Image viewer

This notebook is for inspecting timelapse microscopy data, with associated sinhgle-cell labels and tracks, showing the infection of human macrophages with Mycobacterium Tuberculosis (Mtb), acquired on an Opera Phenix confocal microscope. 

In [1]:
import napari
import os, glob
from macrohet import dataio #, visualise, notify
import numpy as np
import os
import re
import numpy as np
import cv2
import btrack
import zarr


### Load experiment of choice

The Opera Phenix is a high-throughput confocal microscope that acquires very large 5-dimensional (TCZXY) images over several fields of view in any one experiment. Therefore, a lazy-loading approach is chosen to mosaic, view and annotate these images. This approach depends upon Dask and DaskFusion. The first step is to load the main metadata file (typically called `Index.idx.xml` and located in the main `Images` directory) that contains the image filenames and associated TCXZY information used to organise the images.

In [2]:
%%time
expt_ID = 'PS0000'

base_dir = f'/mnt/SYNO/macrohet_syno/data/{expt_ID}/'
# base_dir = f'/mnt/DATA/macrohet/{expt_ID}/'

metadata_fn = glob.glob(os.path.join(base_dir, 'acquisition/Images/Index*xml'))[0]
metadata = dataio.read_harmony_metadata(metadata_fn)  
metadata

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!
CPU times: user 11.4 s, sys: 976 ms, total: 12.4 s
Wall time: 13 s


,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0303K1F1P1R1,Ok,r03c03f01p01-ch1sk1fk1fl1.tiff,3,3,1,1,0,1,1,...,0,0.135583505,0,2021-04-16T19:09:33.84+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
1,0303K1F1P1R2,Ok,r03c03f01p01-ch2sk1fk1fl1.tiff,3,3,1,1,0,2,1,...,0,0.135583505,0,2021-04-16T19:09:33.84+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
2,0303K1F1P2R1,Ok,r03c03f01p02-ch1sk1fk1fl1.tiff,3,3,1,2,0,1,1,...,2E-06,0.135585502,0,2021-04-16T19:09:34.12+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
3,0303K1F1P2R2,Ok,r03c03f01p02-ch2sk1fk1fl1.tiff,3,3,1,2,0,2,1,...,2E-06,0.135585502,0,2021-04-16T19:09:34.12+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
4,0303K1F1P3R1,Ok,r03c03f01p03-ch1sk1fk1fl1.tiff,3,3,1,3,0,1,1,...,4E-06,0.135587499,0,2021-04-16T19:09:34.4+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113395,0609K75F9P1R2,Ok,r06c09f09p01-ch2sk75fk1fl1.tiff,6,9,9,1,74,2,1,...,0,0.135533601,266399.61,2021-04-19T21:14:19.477+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
113396,0609K75F9P2R1,Ok,r06c09f09p02-ch1sk75fk1fl1.tiff,6,9,9,2,74,1,1,...,2E-06,0.135535598,266399.61,2021-04-19T21:14:19.757+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
113397,0609K75F9P2R2,Ok,r06c09f09p02-ch2sk75fk1fl1.tiff,6,9,9,2,74,2,1,...,2E-06,0.135535598,266399.61,2021-04-19T21:14:19.757+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
113398,0609K75F9P3R1,Ok,r06c09f09p03-ch1sk75fk1fl1.tiff,6,9,9,3,74,1,1,...,4E-06,0.135537595,266399.61,2021-04-19T21:14:20.037+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."


### View assay layout and mask information (optional)

The Opera Phenix acquires many time lapse series from a range of positions. The first step is to inspect the image metadata, presented in the form of an `Assaylayout/experiment_ID.xml` file, to show which positions correspond to which experimental assays.

In [3]:
metadata_path = glob.glob(os.path.join(base_dir, 'acquisition/Assaylayout/*.xml'))[0]
assay_layout = dataio.read_harmony_metadata(metadata_path, assay_layout=True,replicate_number=False)# mask_exist=True,  image_dir = image_dir, image_metadata = metadata)
assay_layout

Reading metadata XML file...
Extracting metadata complete!


Strain Compound Concentration ConcentrationEC
Row Column                                              
3   4         RD1     CTRL             0             EC0
    5          WT     CTRL             0             EC0
    6          WT      PZA            60            EC50
    7          WT      RIF           0.1            EC50
    8          WT      INH          0.04            EC50
    9          WT      BDQ          0.02            EC50
4   4         RD1     CTRL             0             EC0
    5          WT     CTRL             0             EC0
    6          WT      PZA            60            EC50
    7          WT      RIF           0.1            EC50
    8          WT      INH          0.04            EC50
    9          WT      BDQ          0.02            EC50
5   4         RD1     CTRL             0             EC0
    5          WT     CTRL             0             EC0
    6          WT      PZA           400            EC99
    7          WT      RIF             2            EC99
    8          WT      INH             2            EC99
    9          WT      BDQ           2.5            EC99
6   4         RD1     CTRL             0             EC0
    5          WT     CTRL             0             EC0
    6          WT      PZA           400            EC99
    7          WT      RIF             2            EC99
    8          WT      INH             2            EC99
    9          WT      BDQ           2.5            EC99

### Load using Zarr

In [4]:
acq_ID = (3, 5)

In [5]:
image_dir = os.path.join(base_dir, f'acquisition/zarr/{acq_ID}.zarr')

zarr_group = zarr.open(image_dir, mode='r')

In [7]:
%%time
images = zarr_group.images[:].max(axis=2)
images.shape

CPU times: user 30.1 s, sys: 32.9 s, total: 1min 2s
Wall time: 2min 57s


(75, 2, 6048, 6048)

#### Load downscaled version from mp4 videos

In [11]:
mp4_video_height

0

In [8]:
mp4_path = glob.glob(f'/mnt/SYNO/videos/macrohet_videos/{expt_ID}/{expt_ID}_{acq_ID}*')[0]

In [14]:
mp4_path#


'/mnt/SYNO/macrohet_syno/data/ND0005/acquisition/mp4/(3, 4).mp4'

In [ ]:
/mnt/SYNO/macrohet_syno/data/ND0005/acquisition/mp4/(3, 4)

In [28]:
mp4_video_width / 1200

1.6

In [44]:
# Load the MP4 video to get its width and height (assuming square frames)
mp4_path = f'/mnt/SYNO/macrohet_syno/data/{expt_ID}/acquisition/mp4/{acq_ID}/{acq_ID}.mp4'

# Open the video using OpenCV
cap = cv2.VideoCapture(mp4_path)

# Get video dimensions (assume width = height)
mp4_video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
mp4_video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Compute scale factor to rescale the tracks to fit the MP4 dimensions
scale_factor =  mp4_video_width / 1200

# Scale tuple for napari tracks
track_scale = (1, scale_factor, scale_factor)  # Assuming time is not scaled

# Print the computed scale tuple
print(f"Scale for napari tracks: {track_scale}")

# Code to load MP4 video frames as a stack of images
frames = []
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Convert the frame to grayscale (if needed)
    # frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Uncomment for grayscale
    frames.append(frame)

# Convert list of frames to a NumPy array (stack of images)
images_mp4 = np.stack(frames, axis=0)

# Release the video capture object
cap.release()

Scale for napari tracks: (1, 1.6, 1.6)


In [42]:
from tqdm.auto import tqdm

In [45]:
# Initialize an empty list to store the rescaled masks
rescaled_segmentations = []

# Iterate through the segmentation stack, rescaling each frame
for i in tqdm(range(segmentation.shape[0]), total = segmentation.shape[0], desc="Rescaling segmentations"):
    # Extract the 2D mask for the current time point
    mask = segmentation[i]
    
    # Rescale the mask to match the MP4 video dimensions
    rescaled_mask = cv2.resize(mask, (mp4_video_width, mp4_video_height), interpolation=cv2.INTER_NEAREST)
    
    # Append the rescaled mask to the list
    rescaled_segmentations.append(rescaled_mask)

# Convert the list of rescaled masks back into a NumPy array (3D stack)
rescaled_stack = np.stack(rescaled_segmentations, axis=0)

# Now `rescaled_stack` contains the resized segmentations that match the MP4 video dimensions
print(f"Rescaled segmentation stack shape: {rescaled_stack.shape}")


Rescaling segmentations:   0%|          | 0/192 [00:00<?, ?it/s]

Rescaled segmentation stack shape: (192, 1920, 1920)


In [17]:
viewer = napari.Viewer(title = f'{expt_ID, acq_ID}, max_proj')


viewer.add_image(images_mp4
                )

<Image layer 'images_mp4' at 0x7f5106f42620>

# Launch napari

In [8]:
viewer = napari.Viewer(title = f'{expt_ID, acq_ID}, max_proj')


viewer.add_image(images,
                 channel_axis = 1, 
                 colormap=['magenta', 'green'],
                 blending = 'additive', 
                 contrast_limits=[[350, 1000],[0, 7000]]
                )

[<Image layer 'Image' at 0x7f6de043c7c0>,
 <Image layer 'Image [1]' at 0x7f6de035fee0>]

In [10]:
with btrack.io.HDF5FileHandler(os.path.join(f'/mnt/SYNO/macrohet_syno/data/{expt_ID}/labels/macrohet_seg_model/{acq_ID}.h5'), #cpv3 
                                           'r', 
                                           obj_type='obj_type_1'
                                           ) as reader:
                segmentation = reader.segmentation
                tracks = reader.tracks

[INFO][2024/12/18 01:44:41 pm] Opening HDF file: /mnt/SYNO/macrohet_syno/data/PS0000/labels/macrohet_seg_model/(3, 5).h5...
[INFO][2024/12/18 01:44:51 pm] Loading segmentation (75, 6048, 6048)
[INFO][2024/12/18 01:44:51 pm] Loading tracks/obj_type_1
[INFO][2024/12/18 01:44:51 pm] Loading LBEP/obj_type_1
[INFO][2024/12/18 01:44:51 pm] Loading objects/obj_type_1 (41424, 5) (41424 filtered: None)
[INFO][2024/12/18 01:44:52 pm] Closing HDF file: /mnt/SYNO/macrohet_syno/data/PS0000/labels/macrohet_seg_model/(3, 5).h5


### Add tracks

In [12]:
filtered_tracks = [track for track in tracks if len(track) > 35]

In [11]:
tracks_scale = (5.04, 5.04) # (1.6, 1.6)
# recoloured_seg = btrack.utils.update_segmentation(segmentation, filtered_tracks, scale = tracks_scale)

In [12]:
# napari_tracks, _, _ = btrack.utils.tracks_to_napari(filtered_tracks, ndim = 2)
napari_tracks, properties, graph = btrack.utils.tracks_to_napari(tracks, ndim = 2)

In [13]:
# tracks_scale = (1, 5.04, 5.04)
viewer.add_tracks(napari_tracks, properties=properties, scale=tracks_scale, name = 'tracks', color_by='t')

/home/dayn/miniconda3/envs/godspeed/lib/python3.10/site-packages/napari/layers/tracks/tracks.py:652: UserWarning: Previous color_by key 't' not present in features. Falling back to track_id
  warn(


<Tracks layer 'tracks' at 0x7f6e0d592650>

### Add segmentation

In [48]:
viewer.add_labels(rescaled_stack) #segmentation)
viewer.add_labels(recoloured_seg)

<Labels layer 'recoloured_seg' at 0x7f508fa98ee0>

In [28]:
viewer.add_image(images_max_proj[-5],
                 channel_axis = 0, 
                 colormap=['green', 'magenta'],
                 blending = 'additive', 
                 contrast_limits=[[0, 2400],[0, 1000]]
                )

[<Image layer 'Image [4]' at 0x7fa481516460>,
 <Image layer 'Image [5]' at 0x7fa4823087f0>]

In [ ]:
viewer.camera.reset()

In [48]:
from macrohet import visualise
visualise.highlight_cell(586, viewer, tracks, scale_factor=5.02)

/home/dayn/analysis/macrohet/macrohet/visualise.py:454: FutureWarning: Argument 'edge_width' is deprecated, please use 'border_width' instead. The argument 'edge_width' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  highlight = viewer.add_points(points, size=size,
/home/dayn/miniconda3/envs/godspeed/lib/python3.10/site-packages/napari/utils/migrations.py:101: FutureWarning: Argument 'edge_color' is deprecated, please use 'border_color' instead. The argument 'edge_color' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  return func(*args, **kwargs)


<Points layer 'cell 586' at 0x7f6dc26ce530>